# Godot 生态仓库 · 颜色矩阵

数据来自本仓库 `godot.db`(5 个 topic、`star ≥ 1000`、最近 3 个月有推送,去重后 51 个仓库)。

矩阵维度:**full_name / url / stars / forks / open_issues / 月均新增 star**。

> 月均新增 star = `stars ÷ 从创建到最后更新的月数`,衡量仓库历史上的平均涨星速度。
> 数值列按列做热力着色;着色用每个单元格的 inline 样式,**GitHub 在线查看即可看到颜色**。

In [1]:
import sqlite3
import pandas as pd
import matplotlib
from IPython.display import HTML

conn = sqlite3.connect('godot.db')
df = pd.read_sql_query(
    'SELECT full_name, url, stars, forks, open_issues, created_at, updated_at FROM repos',
    conn,
)
conn.close()

# 月均新增 star = stars / 从创建到最后更新的月数(至少按 1 个月计,避免除零)
created = pd.to_datetime(df['created_at'], utc=True)
updated = pd.to_datetime(df['updated_at'], utc=True)
active_months = ((updated - created).dt.days / 30.44).clip(lower=1.0)
df['stars_per_month'] = (df['stars'] / active_months).round(1)
df = (df.drop(columns=['created_at', 'updated_at'])
        .sort_values('stars', ascending=False)
        .reset_index(drop=True))
len(df)

51

In [2]:
# 把数值按列归一化映射到 colormap,生成 inline background-color(rgba,半透明保证文字可读)
NUM_COLS = {
    'stars': 'YlGnBu',
    'forks': 'OrRd',
    'open_issues': 'Purples',
    'stars_per_month': 'Greens',
}
ALPHA = 0.45

def _cell_bg(value, vmin, vmax, cmap_name):
    if vmax == vmin:
        t = 0.0
    else:
        t = (value - vmin) / (vmax - vmin)
    r, g, b, _ = matplotlib.colormaps[cmap_name](t)
    return f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{ALPHA})'

def render_matrix(frame, caption):
    bounds = {c: (frame[c].min(), frame[c].max()) for c in NUM_COLS if c in frame.columns}
    head_style = ('background-color:#222;color:#fff;padding:6px 10px;'
                  'text-align:left;position:sticky;top:0')
    html = [f'<table style="border-collapse:collapse;font-family:system-ui,Arial,sans-serif;'
            f'font-size:13px"><caption style="font-size:16px;font-weight:bold;'
            f'padding:8px;text-align:left">{caption}</caption>']
    # 表头
    html.append('<tr>' + ''.join(
        f'<th style="{head_style}">{col}</th>' for col in frame.columns) + '</tr>')
    # 数据行
    for _, row in frame.iterrows():
        cells_html = []
        for col in frame.columns:
            v = row[col]
            base = 'padding:4px 10px;border-bottom:1px solid #eee'
            if col == 'full_name':
                cells_html.append(f'<td style="{base};font-weight:bold">{v}</td>')
            elif col == 'url':
                cells_html.append(
                    f'<td style="{base}"><a href="{v}" target="_blank">link</a></td>')
            elif col in NUM_COLS:
                vmin, vmax = bounds[col]
                bg = _cell_bg(v, vmin, vmax, NUM_COLS[col])
                txt = f'{v:,.1f}' if col == 'stars_per_month' else f'{int(v):,}'
                cells_html.append(
                    f'<td style="{base};background-color:{bg};text-align:right">{txt}</td>')
            else:
                cells_html.append(f'<td style="{base}">{v}</td>')
        html.append('<tr>' + ''.join(cells_html) + '</tr>')
    html.append('</table>')
    return HTML('\n'.join(html))

render_matrix(df, 'Godot 生态仓库颜色矩阵 — 按 stars 降序')

full_name,url,stars,forks,open_issues,stars_per_month
godotengine/godot,link,"112,899","25,726","18,480",755.1
Donchitos/Claude-Code-Game-Studios,link,"22,010","3,181",44,"5,153.7"
heroiclabs/nakama,link,"12,778","1,430",124,112.9
godotengine/awesome-godot,link,"10,210",549,53,76.0
Orama-Interactive/Pixelorama,link,"9,771",516,83,119.0
0xFA11/MultiplayerNetworkingResources,link,"8,556",534,2,84.8
Redot-Engine/redot-engine,link,"5,880",306,61,56.4
dialogic-godot/dialogic,link,"5,726",335,168,79.0
RodZill4/material-maker,link,"5,554",352,311,58.5
godotengine/godot-docs,link,"5,430","3,686","1,077",43.0


## 维度速览

- **stars / forks / open_issues**:GitHub 即时指标,采集时快照。
- **stars_per_month(月均新增 star)**:用 `stars` 除以「创建→最后更新」跨度的月数,近似该仓库生命周期内的平均涨星速度。注意这是历史平均,不等于当前热度——老牌项目即便现在仍火,均值也会被早期低速期摊薄;新项目则可能偏高。

In [3]:
# 月均涨星 Top 15(换个角度看“增长效率”而非绝对体量)
top_growth = (df.sort_values('stars_per_month', ascending=False)
                .head(15)
                [['full_name', 'url', 'stars', 'stars_per_month']]
                .reset_index(drop=True))
render_matrix(top_growth, '月均新增 star Top 15')

full_name,url,stars,stars_per_month
Donchitos/Claude-Code-Game-Studios,link,"22,010","5,153.7"
htdt/godogen,link,"3,583",807.9
godotengine/godot,link,"112,899",755.1
MattParkerDev/SharpIDE,link,"3,774",510.6
KsanaDock/Microverse,link,"2,375",283.5
Coding-Solo/godot-mcp,link,"4,319",273.9
Orama-Interactive/Pixelorama,link,"9,771",119.0
heroiclabs/nakama,link,"12,778",112.9
godot-rust/gdext,link,"4,904",103.2
TokisanGames/Terrain3D,link,"3,988",97.4
